In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch12-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "12-06-active-learning.py"

# 拿保留骨架評估主動學習

執行分析格會寫出 Python 程式，最後一格執行並顯示圖。試做練習時，可將程式接在分析格末尾再執行這兩格。

In [ ]:
%%writefile "{script}"
from collections import defaultdict
from pathlib import Path
import os

import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.preprocessing import StandardScaler

# ESOL: 1,128 measurements of log10(S / (mol/L)), DOI 10.1021/ci034243x.
_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
records = defaultdict(list)
for smi, value, weight in zip(data.ids, data.y[:, 0], data.w[:, 0]):
    if weight > 0 and np.isfinite(value):
        records[Chem.MolToSmiles(Chem.MolFromSmiles(smi))].append(float(value))
smiles = np.array(sorted(records))

def reveal(i):
    return np.mean(records[smiles[i]])

def descriptors(smi):
    mol = Chem.MolFromSmiles(smi)
    return [Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol)]

def fit_gp(indices):
    model = GaussianProcessRegressor(kernel=Matern(length_scale=2, nu=2.5),
                                     alpha=0.1, optimizer=None, normalize_y=True)
    return model.fit(X[indices], [reveal(i) for i in indices])

out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
from sklearn.metrics import root_mean_squared_error
from rdkit.Chem.Scaffolds import MurckoScaffold

# 同一骨架的分子一起分配。索引按骨架大小及 SMILES 次序固定。
ds = dc.data.NumpyDataset(X=np.arange(len(smiles))[:, None], ids=smiles)
pool, _, test = dc.splits.ScaffoldSplitter().split(ds, frac_train=0.8, frac_valid=0, frac_test=0.2)
def scaffolds(indices):
    return {MurckoScaffold.MurckoScaffoldSmiles(smiles=smiles[i]) for i in indices}
assert scaffolds(pool).isdisjoint(scaffolds(test))
raw = np.array([descriptors(s) for s in smiles])
X = StandardScaler().fit(raw[pool]).transform(raw)
budgets = np.arange(12, 61, 4)

def learn(rule, seed):
    rng = np.random.default_rng(seed)
    seen = list(rng.choice(pool, budgets[0], replace=False))
    predictions = []
    for n in budgets:
        assert len(seen) == n
        model = fit_gp(seen)
        predictions.append(model.predict(X[test]))
        if n == budgets[-1]:
            break
        left = np.setdiff1d(pool, seen)
        if rule == "uncertain":
            _, sigma = model.predict(X[left], return_std=True)
            batch = left[np.argsort(-sigma, kind="stable")[:4]]
        elif rule == "random":
            batch = rng.choice(left, 4, replace=False)
        else:
            raise ValueError(rule)
        seen.extend(batch)
    return predictions

# 兩組路徑都完成後才以測試標籤評分。
seeds = [7, 17, 27, 37, 42, 57, 67, 77]
paths = {rule: [learn(rule, seed) for seed in seeds] for rule in ["uncertain", "random"]}
test_y = [reveal(i) for i in test]
errors = {rule: np.array([[root_mean_squared_error(test_y, pred) for pred in run]
                          for run in runs]) for rule, runs in paths.items()}
print("pool/test:", len(pool), len(test), "scaffolds:", len(scaffolds(pool)), len(scaffolds(test)))
fig, ax = plt.subplots(figsize=(7.2, 4.2), layout="constrained")
for rule, color in [("uncertain", "#007875"), ("random", "#a75229")]:
    error = errors[rule]
    print(rule, "mean by budget:", error.mean(axis=0).round(4), "final by seed:", error[:, -1].round(4))
    q1, q3 = np.percentile(error, [25, 75], axis=0)
    ax.fill_between(budgets, q1, q3, color=color, alpha=.13)
    ax.plot(budgets, error.mean(axis=0), marker="o", ms=3, color=color, label=rule.capitalize())
ax.set(xlabel="Structures measured", ylabel="Held-out RMSE (log S)",
       title="Predicting ESOL solubility on held-out scaffolds")
ax.legend(title="Mean; band = middle 50%")
fig.savefig(out / "12-06-active-learning.png", dpi=170)


In [ ]:
from IPython.display import Image, display

import os
environment = dict(os.environ, VECLIB_MAXIMUM_THREADS="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MPLBACKEND="Agg")
subprocess.run([python, str(script)], check=True, env=environment)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "12-06-active-learning.png")))